# 01 - PPO

PPO uses one sampled solution per problem, a separate Qwen3 scalar critic, full token-level GAE, symmetric ratio clipping, and value clipping. TRL 1.8 keeps PPO under `trl.experimental.ppo`; the subclass below uses its model/state initialization but overrides the neural-reward training loop so the reward is exact GSM8K answer correctness.

In [ ]:
# Hyperparameters
sft_checkpoint = "./checkpoints/sft_base"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17
train_problems = 256
eval_problems = 64
num_epochs = 1
rollout_batch_size = 128
generation_micro_batch_size = 16
forward_micro_batch_size = 8
ppo_minibatch_size = 32
num_ppo_epochs = 4
max_prompt_tokens = 384
max_completion_tokens = 256
temperature = 0.8
top_p = 1.0
actor_learning_rate = 1e-6
critic_learning_rate = 2e-6
weight_decay = 0.0
policy_clip_epsilon = 0.2
value_clip_epsilon = 0.2
kl_coefficient = 0.05
gamma = 1.0
gae_lambda = 0.95
max_grad_norm = 1.0
eval_every_steps = 1
eval_batch_size = 16
wandb_project = "swe-rl-ablations"
wandb_run_name = "ppo"
output_path = "./checkpoints/ppo"

optimizer_steps = train_problems // rollout_batch_size
assert rollout_batch_size == 128 and train_problems == 256 and eval_problems == 64
assert rollout_batch_size % ppo_minibatch_size == 0
assert ppo_minibatch_size % forward_micro_batch_size == 0
assert policy_clip_epsilon == 0.2 and gamma == 1.0 and gae_lambda == 0.95
print(f"PPO updates: {optimizer_steps}; PPO epochs per rollout: {num_ppo_epochs}")

In [ ]:
from pathlib import Path
import os
import re
import time

import torch
import torch.nn as nn
import wandb
from datasets import Dataset, load_dataset
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, set_seed
from trl.experimental.ppo import PPOConfig, PPOTrainer

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = project_root / sft_checkpoint
if not (checkpoint_path / "config.json").exists():
    raise FileNotFoundError("Run 00_sft.ipynb first.")
os.environ["WANDB_PROJECT"] = wandb_project
set_seed(seed)

train_rows = load_dataset(dataset_id, dataset_config, split="train", revision=dataset_revision).shuffle(seed=seed).select(range(train_problems))
eval_rows = load_dataset(dataset_id, dataset_config, split="test", revision=dataset_revision).shuffle(seed=seed).select(range(eval_problems))

In [ ]:
def gold_number(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K answer: {answer!r}")
    return match.group(1).replace(",", "")


def predicted_number(completion):
    match = re.search(r"Final answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", completion.strip(), re.IGNORECASE)
    return None if not match else match.group(1).replace(",", "")


def exact_reward(completion, answer):
    prediction = predicted_number(completion)
    if prediction is None:
        return -1.0
    return 1.0 if float(prediction) == float(gold_number(answer)) else 0.0


def prompt_text(question):
    messages = [{"role": "user", "content": f"Solve this grade-school math problem. Show your reasoning, then end with `Final answer: <number>`.\n\n{question}"}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def masked_mean(values, mask):
    return (values * mask).sum() / mask.sum().clamp_min(1)


def gradient_norm(parameters):
    norms = [parameter.grad.detach().norm(2) for parameter in parameters if parameter.grad is not None]
    return float(torch.norm(torch.stack(norms), 2).item()) if norms else 0.0


def explained_variance(prediction, target, mask):
    prediction = prediction[mask.bool()]
    target = target[mask.bool()]
    variance = target.var(unbiased=False)
    return 0.0 if variance <= 1e-12 else float((1 - (target - prediction).var(unbiased=False) / variance).item())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

actor = AutoModelForCausalLM.from_pretrained(checkpoint_path, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2")
reference = AutoModelForCausalLM.from_pretrained(checkpoint_path, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2")
critic = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_path,
    num_labels=1,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
critic.config.pad_token_id = tokenizer.pad_token_id
for parameter in reference.parameters():
    parameter.requires_grad_(False)
reference.eval()


class UnusedNeuralReward(nn.Module):
    """PPOTrainer requires a reward module, but exact rewards are computed from decoded answers."""
    def __init__(self):
        super().__init__()
        self.anchor = nn.Parameter(torch.zeros(()), requires_grad=False)

unused_reward_model = UnusedNeuralReward()

In [ ]:
def generate_rollout(model, rows):
    all_input_ids = []
    all_attention_masks = []
    all_completion_masks = []
    all_text = []
    prompts = [prompt_text(question) for question in rows["question"]]
    for start in range(0, len(prompts), generation_micro_batch_size):
        chunk = prompts[start:start + generation_micro_batch_size]
        encoded = tokenizer(
            chunk,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_prompt_tokens,
        ).to(model.device)
        generated = model.generate(
            **encoded,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_completion_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        completion = generated[:, max_prompt_tokens:]
        if completion.shape[1] < max_completion_tokens:
            padding = torch.full(
                (completion.shape[0], max_completion_tokens - completion.shape[1]),
                tokenizer.pad_token_id,
                device=completion.device,
            )
            completion = torch.cat([completion, padding], dim=1)
        eos = completion.eq(tokenizer.eos_token_id)
        first_eos = torch.where(eos.any(1), eos.float().argmax(1), torch.full((len(chunk),), max_completion_tokens - 1, device=completion.device)).long()
        positions = torch.arange(max_completion_tokens, device=completion.device).unsqueeze(0)
        generated_mask = positions <= first_eos.unsqueeze(1)
        input_ids = torch.cat([encoded.input_ids, completion], dim=1)
        attention_mask = torch.cat([encoded.attention_mask, generated_mask.long()], dim=1)
        completion_mask = torch.zeros((len(chunk), input_ids.shape[1] - 1), device=model.device)
        completion_mask[:, max_prompt_tokens - 1:] = generated_mask.float()
        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)
        all_completion_masks.append(completion_mask)
        all_text.extend(tokenizer.batch_decode(completion, skip_special_tokens=True))
    return torch.cat(all_input_ids), torch.cat(all_attention_masks), torch.cat(all_completion_masks), all_text


def token_logprobs(model, input_ids, attention_mask, requires_grad):
    chunks = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_micro_batch_size):
            ids = input_ids[start:start + forward_micro_batch_size]
            mask = attention_mask[start:start + forward_micro_batch_size]
            logits = model(input_ids=ids, attention_mask=mask, use_cache=False).logits[:, :-1]
            chunks.append(torch.log_softmax(logits.float(), -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1))
    return torch.cat(chunks)


def token_values(model, input_ids, attention_mask, requires_grad):
    chunks = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_micro_batch_size):
            ids = input_ids[start:start + forward_micro_batch_size]
            mask = attention_mask[start:start + forward_micro_batch_size]
            output = model(input_ids=ids, attention_mask=mask, output_hidden_states=True, use_cache=False, return_dict=True)
            chunks.append(model.score(output.hidden_states[-1][:, :-1]).squeeze(-1).float())
    return torch.cat(chunks)


def full_gae(token_rewards, values, mask):
    advantages = torch.zeros_like(values)
    running = torch.zeros(len(values), device=values.device)
    for index in range(values.shape[1] - 1, -1, -1):
        next_value = values[:, index + 1] * mask[:, index + 1] if index + 1 < values.shape[1] else 0.0
        delta = token_rewards[:, index] + gamma * next_value - values[:, index]
        running = (delta + gamma * gae_lambda * running) * mask[:, index]
        advantages[:, index] = running
    returns = advantages + values
    selected = advantages[mask.bool()]
    advantages = (advantages - selected.mean()) / (selected.std(unbiased=False) + 1e-8)
    advantages = advantages * mask
    return advantages, returns

The trainer below keeps PPOTrainer's official initialization and public `.train()` entry point. The override is explicit because the stock implementation assumes a neural reward model and combines actor/critic optimization, which would make the requested exact reward and separate timings dishonest.

In [ ]:
class ExactAnswerPPOTrainer(PPOTrainer):
    def __init__(self, *args, exact_train_rows, exact_eval_rows, **kwargs):
        super().__init__(*args, **kwargs)
        self.exact_train_rows = exact_train_rows
        self.exact_eval_rows = exact_eval_rows

    @torch.no_grad()
    def exact_evaluate(self):
        policy = self.accelerator.unwrap_model(self.model).policy
        policy.eval()
        rewards = []
        for start in range(0, len(self.exact_eval_rows), eval_batch_size):
            stop = min(start + eval_batch_size, len(self.exact_eval_rows))
            rows = self.exact_eval_rows.select(range(start, stop))
            prompts = [prompt_text(question) for question in rows["question"]]
            encoded = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_prompt_tokens).to(policy.device)
            generated = policy.generate(**encoded, do_sample=False, max_new_tokens=max_completion_tokens, pad_token_id=tokenizer.pad_token_id)
            text = tokenizer.batch_decode(generated[:, encoded.input_ids.shape[1]:], skip_special_tokens=True)
            rewards.extend(exact_reward(completion, answer) for completion, answer in zip(text, rows["answer"]))
        policy.train()
        return sum(rewards) / len(rewards)

    def train(self):
        wrapped = self.accelerator.unwrap_model(self.model)
        policy = wrapped.policy
        value_model = wrapped.value_model
        policy.train()
        value_model.train()
        policy_optimizer = torch.optim.AdamW(policy.parameters(), lr=actor_learning_rate, weight_decay=weight_decay)
        critic_optimizer = torch.optim.AdamW(value_model.parameters(), lr=critic_learning_rate, weight_decay=weight_decay)
        run = wandb.init(project=wandb_project, name=wandb_run_name)
        torch.cuda.reset_peak_memory_stats()

        global_step = 0
        for start in range(0, len(self.exact_train_rows), rollout_batch_size):
            step_started = time.perf_counter()
            rows = self.exact_train_rows.select(range(start, start + rollout_batch_size))
            with torch.no_grad():
                input_ids, attention_mask, completion_mask, completions = generate_rollout(policy, rows)
                reward = torch.tensor(
                    [exact_reward(completion, answer) for completion, answer in zip(completions, rows["answer"])],
                    device=policy.device,
                )
                old_logps = token_logprobs(policy, input_ids, attention_mask, requires_grad=False)
                reference_logps = token_logprobs(self.ref_model, input_ids, attention_mask, requires_grad=False)
                old_values = token_values(value_model, input_ids, attention_mask, requires_grad=False)
                token_rewards = -kl_coefficient * (old_logps - reference_logps) * completion_mask
                final_positions = (torch.arange(completion_mask.shape[1], device=policy.device).unsqueeze(0) * completion_mask.long()).max(1).values
                token_rewards[torch.arange(len(reward), device=policy.device), final_positions] += reward
                advantages, returns = full_gae(token_rewards, old_values, completion_mask)

            policy_losses = []
            critic_losses = []
            clipped_tokens = []
            policy_grad_norms = []
            critic_grad_norms = []
            policy_seconds = 0.0
            critic_seconds = 0.0

            for _ in range(num_ppo_epochs):
                permutation = torch.randperm(rollout_batch_size, device=policy.device)
                for minibatch_start in range(0, rollout_batch_size, ppo_minibatch_size):
                    indices = permutation[minibatch_start:minibatch_start + ppo_minibatch_size]

                    torch.cuda.synchronize()
                    started_at = time.perf_counter()
                    new_logps = token_logprobs(policy, input_ids[indices], attention_mask[indices], requires_grad=True)
                    ratio = torch.exp(new_logps - old_logps[indices])
                    unclipped = ratio * advantages[indices]
                    clipped = ratio.clamp(1 - policy_clip_epsilon, 1 + policy_clip_epsilon) * advantages[indices]
                    policy_loss = -masked_mean(torch.minimum(unclipped, clipped), completion_mask[indices])
                    policy_optimizer.zero_grad(set_to_none=True)
                    policy_loss.backward()
                    policy_grad_norms.append(gradient_norm(policy.parameters()))
                    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
                    policy_optimizer.step()
                    torch.cuda.synchronize()
                    policy_seconds += time.perf_counter() - started_at
                    controlling_clip = ((advantages[indices] > 0) & (ratio > 1 + policy_clip_epsilon)) | ((advantages[indices] < 0) & (ratio < 1 - policy_clip_epsilon))
                    clipped_tokens.append(float(masked_mean(controlling_clip.float(), completion_mask[indices]).item()))
                    policy_losses.append(float(policy_loss.item()))

                    torch.cuda.synchronize()
                    started_at = time.perf_counter()
                    new_values = token_values(value_model, input_ids[indices], attention_mask[indices], requires_grad=True)
                    clipped_values = old_values[indices] + (new_values - old_values[indices]).clamp(-value_clip_epsilon, value_clip_epsilon)
                    value_loss = 0.5 * masked_mean(
                        torch.maximum((new_values - returns[indices]).square(), (clipped_values - returns[indices]).square()),
                        completion_mask[indices],
                    )
                    critic_optimizer.zero_grad(set_to_none=True)
                    value_loss.backward()
                    critic_grad_norms.append(gradient_norm(value_model.parameters()))
                    torch.nn.utils.clip_grad_norm_(value_model.parameters(), max_grad_norm)
                    critic_optimizer.step()
                    torch.cuda.synchronize()
                    critic_seconds += time.perf_counter() - started_at
                    critic_losses.append(float(value_loss.item()))

            global_step += 1
            final_values = token_values(value_model, input_ids, attention_mask, requires_grad=False)
            post_update_logps = token_logprobs(policy, input_ids, attention_mask, requires_grad=False)
            metrics = {
                "train/reward": float(reward.mean().item()),
                "loss/policy": sum(policy_losses) / len(policy_losses),
                "objective/kl_sft": float(masked_mean(post_update_logps - reference_logps, completion_mask).item()),
                "policy/tokens_clipped_or_masked_pct": 100 * sum(clipped_tokens) / len(clipped_tokens),
                "critic/loss": sum(critic_losses) / len(critic_losses),
                "critic/explained_variance": explained_variance(final_values, returns, completion_mask),
                "grad_norm/policy": sum(policy_grad_norms) / len(policy_grad_norms),
                "grad_norm/critic": sum(critic_grad_norms) / len(critic_grad_norms),
                "time/policy_forward_backward_s": policy_seconds,
                "time/critic_forward_backward_s": critic_seconds,
                "time/step_s": time.perf_counter() - step_started,
                "gpu/peak_memory_gb": torch.cuda.max_memory_allocated() / 1024**3,
            }
            if global_step % eval_every_steps == 0 or global_step == optimizer_steps:
                metrics["eval/reward"] = self.exact_evaluate()
            run.log(metrics, step=global_step)

        policy.save_pretrained(project_root / output_path)
        tokenizer.save_pretrained(project_root / output_path)
        run.finish()
        return self.state

In [ ]:
tokenized_prompts = train_rows.map(
    lambda row: tokenizer(prompt_text(row["question"]), truncation=True, max_length=max_prompt_tokens),
    remove_columns=train_rows.column_names,
)

ppo_args = PPOConfig(
    output_dir=str(project_root / output_path),
    run_name=wandb_run_name,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=rollout_batch_size,
    gradient_accumulation_steps=1,
    response_length=max_completion_tokens,
    temperature=temperature,
    learning_rate=actor_learning_rate,
    num_ppo_epochs=num_ppo_epochs,
    num_mini_batches=rollout_batch_size // ppo_minibatch_size,
    cliprange=policy_clip_epsilon,
    cliprange_value=value_clip_epsilon,
    kl_coef=kl_coefficient,
    gamma=gamma,
    lam=gae_lambda,
    report_to=[],
    bf16=True,
)
trainer = ExactAnswerPPOTrainer(
    args=ppo_args,
    processing_class=tokenizer,
    model=actor,
    ref_model=reference,
    reward_model=unused_reward_model,
    train_dataset=tokenized_prompts,
    value_model=critic,
    exact_train_rows=train_rows,
    exact_eval_rows=eval_rows,
)
trainer.train()